# Simple Agentic RAG

This notebook walks through building an Agentic RAG system using LangChain and FAISS.

**Steps:**
1. Install dependencies
2. Ingest documents into a FAISS vector store
3. Build a LangChain agent with a document search tool
4. Ask questions

## 1. Install Dependencies

In [1]:
import os

if not os.path.exists('requirements.txt'):
    !wget https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-langgraph-02/requirements.txt
else:
    print('requirements.txt already exists.')
%pip install -r  requirements.txt -q


requirements.txt already exists.


## 2. Set API Keys

In [3]:
import os
from dotenv import load_dotenv

# Fallback: try Google Colab secrets
try:
    from google.colab import userdata
    if not os.getenv("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if not os.getenv("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY") or ""
except ImportError:
    # Load from .env file (local development)
    load_dotenv("/home/sirkumar/FDP-AGENENTIC-AI-RAG/.env")

HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip()
GEMINI_API_KEY = (os.getenv("GEMINI_API_KEY") or "").strip()


if not GEMINI_API_KEY or not HF_TOKEN:
    print("Warning: HF_TOKEN or GEMINI_API_KEY not set. Please set it in .env file or Colab secrets.")

## 3. Ingest Documents into FAISS Vector Store

Defines knowledge inline, splits it into chunks, embeds them, and saves the vector store locally.

In [4]:
department_info = [
    # Department overview
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.",
    "Students can choose from 5 courses ranging from core subjects to electives and hands-on project work, with strong industry exposure.",
    "DQE offers a postgraduate module 'Foundations of Quantum AI' and an undergraduate elective 'Quantum Machine Learning' using IBM Qiskit and PennyLane.",
    "All students have access to cloud quantum hardware through IBM Quantum Network and Amazon Braket as part of their coursework.",

    # Quantum AI research
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.",
    "A key research thread is Quantum Reinforcement Learning (QRL), where quantum agents learn policies faster than classical counterparts on specific problem classes.",
    "Project QuLearn is DQE's flagship initiative building hybrid classical-quantum models for drug discovery and materials science.",

]


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

documents = [Document(page_content=text) for text in department_info]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
docs = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2")
db = FAISS.from_documents(docs, embeddings)
db.save_local("vectorstore")

print(f"Vector DB created successfully ({len(docs)} chunks)")

/tmp/ipykernel_4696/3139761478.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector DB created successfully (8 chunks)


## 4. Build the RAG Agent

Loads the vector store and wires it up as a tool for a LangChain ZERO_SHOT_REACT agent.

In [6]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS

db = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True,
)

@tool
def search_answers(query: str) -> str:
    """Search the knowledge base"""
    results = db.similarity_search(query, k=2)
    return "\n".join([doc.page_content for doc in results])

# ── Switch providers by setting MODEL_NAME — no code changes needed ──────────
#   google_genai → MODEL_NAME=google_genai:gemini-2.5-flash   (default)
#   openai       → MODEL_NAME=openai:gpt-4o
#   anthropic    → MODEL_NAME=anthropic:claude-3-5-sonnet-20241022
#   huggingface  → MODEL_NAME=huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0
#   ollama       → MODEL_NAME=ollama:llama3
# ────────────────────────────────────────────────────────────────────────────

# Make loaded keys visible to init_chat_model via standard env-var names
os.environ.setdefault("GOOGLE_API_KEY", GEMINI_API_KEY or "")

MODEL_NAME = os.getenv("MODEL_NAME", "google_genai:gemini-2.5-flash")
chat_llm = init_chat_model(MODEL_NAME, temperature=0.1)

agent = create_agent(
    model=chat_llm,
    tools=[search_answers],
    system_prompt=(
        "You are an intelligent Agentic RAG system. "
        "Decide whether document retrieval is needed. "
        "If needed, use the search_answers tool."
    ),
)

print(f"Agent ready (model: {MODEL_NAME}).")

Agent ready (model: google_genai:gemini-2.5-flash).


## 5. Ask Questions

Edit `question` below and re-run this cell to query the agent.

In [7]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print("\nAnswer:", result["messages"][-1].content)


ask("What is DQE?")


Answer: [{'type': 'text', 'text': 'DQE stands for the Department of Quantum Engineering. Its primary research focus is Quantum AI, which is the intersection of quantum computing and artificial intelligence. The department has approximately 140 students and 20 professors, and they concentrate on applied quantum computing and intelligent systems.', 'extras': {'signature': 'CqEEAQw51sen6gJfcpc/UoNWSpJ8yS8db1qg1YlyApsUj0DD2azLxrm7p6mN2mwPjJqNMpONMUaQr4at1w/mrzV4fOM+/Y+oMhp9Cb92zTMftG2o+YOfrS1TqXmtxCciqR3D9zLqhuOvxZEFubLp4YCEGJ2onIHCck8Yzbau/51X0nClPD0G7CB2aaxMv9mdhD96cGKMr8Iim1Q1h8GFCqmxPajdOx+Uv5TZSl6aXxdMqxx5SZMo85BrSKQZw2VFxYFoWQHd/ig9OspkcupAn8Kvl8+8r6M5GlNqQStFEI9Ji3JeS/SJ0WBhFmVclFg4nHqzwVe0QcA3Kl20cnAD7B9kn36ZrqfsheNoyWaYjNOw6w/KN1kIxQzQvrGN8eoE6RM66qWZEzVdEx+boAKYcAwqiV+xiye+J9U/uAXLSrgKGhkWplIE1X8loit+F32EJlYgo3s0DDbgrbWK+O1mC+wJPsd55+R4T+fgZpL1hZ5cldvDVnmFDdWMH0CM2FQvZFAu+psdOaBchcOdcs2T3SgxispSxzbq9S+zjWahpXVnNlx74t6ECJ9EdD8trOgHOfFYWqsntMUl1ZHSROSLzqhOv/kzJjpLMM8Losh9phoCS4oAv

In [8]:
ask("How many students does DADQEQ have?")


Answer: [{'type': 'text', 'text': 'The Department of Quantum Engineering (DQE) has around 140 students.', 'extras': {'signature': 'CrIFAQw51sf92tvlPD4dK8G6AiES8T8+v2MsiGFkIMEc6+exdElGrhz7y07d61TDCE23ZYJXGBz54LmvnyXJNMwx/iZa5KTQSN9KhVWvUMct+9pst9UaAUGLWQA8bSmwtbqKWBbmYoP1l8a5UzGl61+j/pwRwiISiaoTWV6L/lkLVafL3YdeYTrrjjqlbSjPGmh+M7ozOprYBzCD0ZOYTGWRG1/HLeoMbSociW0PfdyuNe4eaC80NDg2gfQj6X1sXXT94NlvHkLDvUWdUAJdeMh7rJn7AqaFvnsVcgtsbAcldmX+h9OUMEDy3EfVVo0SxSFV/PLeK7FjUmmptIHa1W7l/EbXKCLvFnrVHd7nSxZISjN+IB9TA/JxbTYdYIZkmn+wkgFmF63gjKumCyvNaQI33qSB+CnWGpbtsV1dKB0G2L7Hdu89ogSoSaORiS9dpGM+VKdVyxXfVyWPOMSgRaMXGJyBswpr4GYhAiY4kt91Mf0+8jmpKjlnDv6fp/RDxCz4PC2zZd6WfgXjA0n9dgtvB+AUjb57JOCmLzIKjHCQD0oi2Ye9jOxx96RsuEKddUlHTZhZJ8H3bRkYEMkqld7uFig1/3pDEDVHlr3Xu+/m2HMmb6q7ieioAT9ytPJMWOfPXpxW4DRdEADCCra/Xxv/KVWPr1ZdcNtU8pKWdVgNHcgz97scCcdZew7pVtZKzOtrxMOKfobawJlq2mn8cb1JMj70vO9JLXgs2NC0R0QGodWVNCB01XhtGUA4AI0TkP9xwd5aeJgAxNATtHpFxQdGaOuk6Jk4g9QXTshidt+ycEbl80qSCmAufJ/v1D6uYjQUj1RNonVP9DcrsyMx2Xb+T34Smr4iy4kp

In [9]:
ask("How many stuents does it have")


Answer: [{'type': 'text', 'text': 'The Department of Quantum Engineering (DQE) has around 140 students.', 'extras': {'signature': 'Cv4EAQw51se0N/hWDxuGoGxgi+Y4HaH2xI4LbOPHc0GptDPsFzyl+7jQJ4kuo3MP2OM2d9o8EWE4Ew1vNZrLK+U7+N3xVFJf2hZtKkFMZHPh11W8ELXR1JcnwHhXBRMxqMoV7AqEv1ILOpdMkS8RAZzLUkUb5a7PjEoAHQHszW5nu0sLViV7eDkQPpC2eISitXDHan+lUAxx7WrwT9dkCbE51a7p6Gfj6kmDbuyOOTrmJu3UA+Ac7+S4sKKc5iABz2zCb7P5vtCpqdSd/e086k8eUT6N1FmU3O2QSsmABfbsB7zvQPhFvDRZaro5MFXdEn8/o9tIlqjNP+Qexmo81oi1M9gtyO2DgWR6gN/I/rAIpa/NoxpaKeZXoLxw7MwiqCnelvgDDxEsnOi/ri+TUfZ/QQBw+Ou2ep3wTxsJCxiyaXQ1UoQnPoJkdUB9qUmUT/4w+IjBAcHrd6VulvmRmYMfKxdYyWWEnPbNVsO7UogMu+sw+qNfKR2YgYHnLbaZjuZM/Qfot9USIs2euw3EL+7kZuI9kIjHGgUvrYdw+XMDdw/NtUdqjvB5x7E5c2C/sKFTmmfVnY2IGDyF6bASCGaRDmcBVSjjJVkdbSzr2B/oXpz6s/zpUG/oxSp1nVWzq2BY0a4OewGkPMMTyhpCrhh7mFaBFXarUk3wvlyvPEARO4NmUpjeR8CICnZLLg0LLroQPgRiOZAvalJJmDQvkfDldyAYCHAKk7htzQDS6yEqNesO38Q2fJiuSDSbno8CrOzH8HzBRIlA/60fr4JSvQ99khtWvNpLe7VL8WZKFFArRvof7UdIOIG0f9JKvW2+JBAmXJ9Noox+sXKP2uQuvas='}}]


## 6. Evaluate with RAGAS

[RAGAS](https://docs.ragas.io/) measures RAG pipeline quality across four metrics:

| Metric | What it measures | Needs ground truth? |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context? | No |
| **Answer Relevancy** | Is the answer relevant to the question? | No |
| **Context Precision** | Are the retrieved chunks actually useful? | Yes |
| **Context Recall** | Did retrieval capture what was needed to answer? | Yes |

Scores range from 0 to 1 — higher is better.

In [10]:
%pip install "ragas>=0.2" -q

## 7. RAGAS

### Retrieval-Augmented Generation As A Service. It's a framework for evaluating the performance of Retrieval-Augmented Generation (RAG) systems.

### What each metric tells you

| Metric | Question it answers | Score guide |
|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved chunks? | < 0.5 → hallucination risk; > 0.8 → trustworthy |
| **Answer Relevancy** | Does the answer actually address the question asked? | < 0.5 → off-topic; > 0.8 → on-point |
| **Context Precision** | Are the top-ranked retrieved chunks the useful ones? | < 0.5 → noisy retrieval; > 0.8 → precise retrieval |
| **Context Recall** | Did retrieval surface everything needed to answer? | < 0.5 → missing evidence; > 0.8 → good coverage |

---

### Reading the scores in this pipeline

**Faithfulness** measures whether the model "sticks to the script." A small model like TinyLlama-1.1B tends to score lower here because it may add plausible-sounding details not present in the retrieved text (hallucination). A cloud model like Gemini typically scores higher because it follows instructions more precisely.

**Answer Relevancy** reflects how well the question is understood. Short, focused answers on a narrow knowledge base (like DISE department info) usually score well. If the agent wanders off-topic or refuses to answer, this drops.

**Context Precision & Recall** are about retrieval quality, not the LLM. Since we use a small FAISS store (5 chunks) with sentence-transformer embeddings, precision is expected to be high — there is little noise to retrieve. Recall may suffer on multi-fact questions if the relevant chunk wasn't ranked in the top-k.

---

### What to do if scores are low

| Symptom | Likely cause | Fix |
|---|---|---|
| Low Faithfulness | LLM adds facts not in context | Use a stronger/instruction-tuned model; tighten the system prompt |
| Low Answer Relevancy | Model misunderstands the question | Improve the system prompt; switch to a better model |
| Low Context Precision | Wrong chunks retrieved | Tune `k`, try MMR retrieval, or re-embed with a better model |
| Low Context Recall | Needed chunk not retrieved | Increase `k`, reduce chunk size, or improve the embedding model |

---

> **Note on the judge LLM:** RAGAS uses the same `judge_llm` to score faithfulness and relevancy. If the judge is also TinyLlama (a small model), scores may be noisy or inconsistent — a stronger model (Gemini, GPT-4o) as the judge gives more reliable scores even when TinyLlama is the RAG agent.

In [11]:
def extract_answer(agent_response: dict) -> str:
    """Pull the plain-text answer out of the agent's message (handles list or str content)."""
    content = agent_response["messages"][-1].content
    if isinstance(content, list):
        return " ".join(
            item.get("text", "") for item in content if isinstance(item, dict)
        ).strip()
    return str(content).strip()


# --- Ground-truth answers (reference) ---
eval_questions = [
    "What is DQE?",
    "How many students does DQE have?",
    "How many professors are in DQE?",
    "What courses are offered in DQE?",
    "What is the primary research focus of DQE?"
]

ground_truths = [
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems. DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "DQE has around 140 students.",
    "DQE has around 20 professors.",
    "DQE offers 5 courses, including core subjects, electives, hands-on project work, a postgraduate module 'Foundations of Quantum AI', and an undergraduate elective 'Quantum Machine Learning'.",
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence. They investigate Quantum Neural Networks (QNNs) and Quantum Reinforcement Learning (QRL), and their flagship initiative is Project QuLearn."
]

# --- Collect RAG answers and retrieved contexts ---
answers, contexts = [], []

for question in eval_questions:
    # Retrieve chunks used as context
    retrieved = db.similarity_search(question, k=2)
    contexts.append([doc.page_content for doc in retrieved])

    # Get the agent's answer
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    answers.append(extract_answer(response))

# Preview
for q, a, ctx in zip(eval_questions, answers, contexts):
    print(f"Q: {q}")
    print(f"A: {a}")
    print(f"Ctx[0]: {ctx[0][:80]}...")
    print()

Q: What is DQE?
A: DQE stands for the Department of Quantum Engineering. Its primary research focus is Quantum AI, which is the intersection of quantum computing and artificial intelligence. The department has approximately 140 students and 20 professors, and they concentrate on applied quantum computing and intelligent systems.
Ctx[0]: DQE's primary research focus is Quantum AI — the intersection of quantum computi...

Q: How many students does DQE have?
A: The Department of Quantum Engineering (DQE) has around 140 students.
Ctx[0]: The Department of Quantum Engineering (DQE) has around 140 students and 20 profe...

Q: How many professors are in DQE?
A: The Department of Quantum Engineering (DQE) has 20 professors.
Ctx[0]: The Department of Quantum Engineering (DQE) has around 140 students and 20 profe...

Q: What courses are offered in DQE?
A: The Department of Quantum Engineering (DQE) offers a postgraduate module called 'Foundations of Quantum AI' and an undergraduate elective call

In [12]:
import sys
from types import ModuleType

# ragas tries to import ChatVertexAI from the stub — add a dummy class to satisfy it
for _mod_name, _attrs in {
    "langchain_community.chat_models.vertexai": ["ChatVertexAI"],
    "langchain_community.chat_models.google_palm": ["ChatGooglePalm"],
    "langchain_community.llms.vertexai": ["VertexAI"],
}.items():
    _mod = sys.modules.setdefault(_mod_name, ModuleType(_mod_name))
    for _attr in _attrs:
        if not hasattr(_mod, _attr):
            setattr(_mod, _attr, type(_attr, (), {}))

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_llm = LangchainLLMWrapper(chat_llm)
judge_embeddings = LangchainEmbeddingsWrapper(embeddings)

samples = [
    SingleTurnSample(
        user_input=q,
        response=a,
        retrieved_contexts=ctx,
        reference=gt,
    )
    for q, a, ctx, gt in zip(eval_questions, answers, contexts, ground_truths)
]
ragas_dataset = EvaluationDataset(samples=samples)

result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        Faithfulness(llm=judge_llm),
        AnswerRelevancy(llm=judge_llm, embeddings=judge_embeddings),
        ContextPrecision(llm=judge_llm),
        ContextRecall(llm=judge_llm),
    ],
)

import pandas as pd

scores_df = result.to_pandas()[
    ["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]
]
scores_df.loc["mean"] = scores_df.mean(numeric_only=True)
scores_df.iloc[-1, 0] = "MEAN"

pd.set_option("display.float_format", "{:.3f}".format)
print(scores_df.to_string(index=False))

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_4696/3602951817.py:16: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_4696/3602951817.py:16: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.c

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

                                user_input  faithfulness  answer_relevancy  context_precision  context_recall
                              What is DQE?         1.000             0.688              1.000           1.000
          How many students does DQE have?         1.000             0.817              1.000           1.000
           How many professors are in DQE?         1.000             0.830              1.000           1.000
          What courses are offered in DQE?         0.000             0.497              0.000           0.000
What is the primary research focus of DQE?         1.000             0.981              0.000           0.500
                                      MEAN         0.800             0.763              0.600           0.700
